In [1]:
import pandas as pd
from pathlib import Path
from collections import defaultdict

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier
import joblib

In [2]:
df = pd.read_csv("C:/Users/wack3/Desktop/docker/mini projet ml/data/football_2025_2027.csv")

In [3]:
cols = ["Season", "Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR","HS", "AS", "HST", "AST", "HC", "AC"]
df = df[cols].copy()

In [4]:
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%Y", errors="coerce")

In [5]:
for col in ["FTHG", "FTAG", "HS", "AS", "HST", "AST", "HC", "AC"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [6]:
df["FTR"] = df["FTR"].astype("string").str.strip().str.upper()

In [7]:
df = df[df["FTR"].isin(["H", "D", "A"])].copy()

In [8]:
df = df.dropna(subset=["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG"])

In [9]:
df = df.drop_duplicates(subset=["Date", "HomeTeam", "AwayTeam"])

In [10]:
df = df.sort_values("Date").reset_index(drop=True)

In [11]:
df.shape

(450, 13)

In [12]:
df["FTR"].value_counts()

FTR
H    190
A    136
D    124
Name: count, dtype: Int64

In [13]:
historique = defaultdict(list)
def calcul_stats(equipe):
    derniers = historique[equipe][-5:]
    n = len(derniers)
    return [
        sum(m["points"] for m in derniers),
        sum(m["win"] for m in derniers),
        sum(m["goals"] for m in derniers) / n if n else 0,
        sum(m["conceded"] for m in derniers) / n if n else 0,
        n
    ]

In [14]:
lignes = []
for _, match in df.iterrows():
    home, away = match["HomeTeam"], match["AwayTeam"]
    lignes.append(calcul_stats(home) + calcul_stats(away))
    result = match["FTR"]
    hp = 3 if result == "H" else (1 if result == "D" else 0)
    ap = 3 if result == "A" else (1 if result == "D" else 0)
    historique[home].append({
        "points": hp, "win": int(result == "H"),
        "goals": match["FTHG"], "conceded": match["FTAG"]
    })
    historique[away].append({
        "points": ap, "win": int(result == "A"),
        "goals": match["FTAG"], "conceded": match["FTHG"]
    })

In [15]:
feature_cols = [
    "Home_Form", "Home_Wins", "Home_Goals_Avg", "Home_Conceded_Avg",
    "Home_Matches_Played", "Away_Form", "Away_Wins", "Away_Goals_Avg",
    "Away_Conceded_Avg", "Away_Matches_Played"
]
features_df = pd.DataFrame(lignes, columns=feature_cols)

In [16]:
X = features_df[feature_cols].copy()
y = df["FTR"].astype(str).copy()

In [17]:
split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

In [18]:
le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [19]:
model = XGBClassifier(
    objective="multi:softprob",
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=3,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=2
)
model.fit(X_train, y_train_encoded)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.85, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=2, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=2,
              num_parallel_tree=None, ...)

In [20]:
y_pred_encoded = model.predict(X_test)

In [21]:
acc_xgb = accuracy_score(y_test_encoded, y_pred_encoded)
acc_xgb 

0.3888888888888889

In [22]:
print("Train Accuracy :", round(model.score(X_train, y_train_encoded), 4))
print("Test Accuracy  :", round(acc_xgb, 4))

Train Accuracy : 0.5444
Test Accuracy  : 0.3889


In [23]:
print("Classification report :")
print(classification_report(y_test_encoded, y_pred_encoded))

Classification report :
              precision    recall  f1-score   support

           0       0.36      0.31      0.33        26
           1       0.25      0.08      0.12        26
           2       0.42      0.66      0.51        38

    accuracy                           0.39        90
   macro avg       0.34      0.35      0.32        90
weighted avg       0.35      0.39      0.35        90



In [24]:
print("Confusion Matrix (A, D, H) :")
print(confusion_matrix(y_test_encoded, y_pred_encoded,
                       labels=range(len(le.classes_))))

Confusion Matrix (A, D, H) :
[[ 8  2 16]
 [ 5  2 19]
 [ 9  4 25]]


In [25]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train_encoded)
acc_dummy = accuracy_score(y_test_encoded, dummy.predict(X_test))

In [26]:
log_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
log_model.fit(X_train, y_train_encoded)
acc_log = accuracy_score(y_test_encoded, log_model.predict(X_test))

In [27]:
print("Baseline majorité :", round(acc_dummy, 4))
print("Logistic Regression:", round(acc_log, 4))
print("XGBoost            :", round(acc_xgb, 4))

Baseline majorité : 0.4222
Logistic Regression: 0.3778
XGBoost            : 0.3889


In [28]:
best_model = model if acc_xgb >= acc_log else log_model
print("Modèle retenu :", "XGBoost" if acc_xgb >= acc_log else "Logistic Regression")

Modèle retenu : XGBoost


In [29]:
model_dir = Path("models") if Path("app.py").exists() else Path("ml_streamlit_docker/models")
model_dir.mkdir(parents=True, exist_ok=True)

In [30]:
joblib.dump(best_model, model_dir / "football_model.pkl")
joblib.dump(le, model_dir / "label_encoder.pkl")
joblib.dump(feature_cols, model_dir / "feature_columns.pkl")
print("Modèle sauvegardé dans :", model_dir.resolve())

Modèle sauvegardé dans : C:\Users\wack3\Desktop\docker\mini projet ml\ml_streamlit_docker\models
